In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_ollama import ChatOllama
model = ChatOllama(model="gemma4:e2b")

agent = create_agent(
    model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on September 25th 2026.")]},
    config
    )

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on September 25th 2026.', additional_kwargs={}, response_metadata={}, id='103a46a6-d649-4efd-b6fe-0ab94daf7e44'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-04-22T14:49:40.8480494Z', 'done': True, 'done_reason': 'stop', 'total_duration': 65020791900, 'load_duration': 1586699700, 'prompt_eval_count': 1110, 'prompt_eval_duration': 6907755000, 'eval_count': 420, 'eval_duration': 54456516900, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--019db5aa-14a2-7802-b382-c8d3883b1ee2-0', tool_calls=[{'name': 'search-flight', 'args': {'departureDate': '25/09/2026', 'flyFrom': 'San Francisco', 'flyTo': 'Tokyo'}, 'id': '4908f141-bc5a-49be-9f37-0cb871505e2c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1110, 'output_tokens': 420, 'total_tokens': 1530}),
              T

In [6]:
print(response["messages"][-1].content)

Please tell me where you would like to fly from, where you want to go, and the departure date so I can search for flights for you.
